# module-modules-iter-isinstance-dispatch — worked example 1: Collect all activation modules into a list using model.modules()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-modules-iter-isinstance-dispatch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

Calling `model.modules()` returns an iterator over every submodule in the network, including the model itself and all nested children, in depth-first order. Paired with `isinstance`, you can pattern-match on the Python type of each module to filter or act on specific layer kinds. This is the standard idiom whenever you need to do something to every layer of a particular type across an arbitrarily deep architecture.

## Worked solution

**Step 1 — iterate with `.modules()`.** `model.modules()` is recursive: it yields the root model, then every direct child, then every grandchild, and so on. You do not need to recurse manually.

**Step 2 — skip non-activation types with `isinstance`.** We want only activation layers. `isinstance(m, (nn.ReLU, nn.LeakyReLU, nn.Sigmoid, nn.Tanh))` returns `True` when `m` belongs to any of those types. A plain `if` (not `elif`) is correct here because we're building an inclusion filter, not a mutually-exclusive dispatch.

**Step 3 — append to the collector list.** Every matching module gets appended. The resulting list can be used to swap activations, inspect inplace flags, etc.

**Why not `model.children()`?** `.children()` only goes one level deep. A deeply nested `nn.Sequential` inside an `nn.Sequential` would have its inner layers invisible to `.children()`. `.modules()` recurses all the way down.

In [ ]:
import torch
import torch.nn as nn

def collect_activation_modules(model: nn.Module) -> list:
    """Return a list of all activation-type submodules found anywhere in model."""
    activation_types = (nn.ReLU, nn.LeakyReLU, nn.Sigmoid, nn.Tanh)
    activations = []
    for m in model.modules():
        if isinstance(m, activation_types):
            activations.append(m)
    return activations

# Exercise it on a small network with nested structure.
net = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Sequential(
        nn.Linear(32, 16),
        nn.LeakyReLU(0.2),
        nn.Linear(16, 8),
        nn.Tanh(),
    ),
    nn.Linear(8, 4),
    nn.Sigmoid(),
)

acts = collect_activation_modules(net)
print(f"Found {len(acts)} activation modules:")
for a in acts:
    print(f"  {type(a).__name__}")
# Expected: ReLU, LeakyReLU, Tanh, Sigmoid (4 total)